# 9 WorkFlow Analista Senior

### 9.5 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán enriquecer
El Analista Sr corre sus scripts en máquinas virtuales de al menos 128 GB de RAM en Toronto, creando una virtual machine para cada corrida.
Estas virtual machines se auto-suicidarán a los 30 minutos de haber terminado de procesar.


## Inicializacion

limpio el ambiente de R


In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# creo PARAM
PARAM <- list()
PARAM$tiempos <- list()
PARAM$tiempos$inicio_notebook <- format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

In [ ]:
#PARAM <- list()
PARAM$semilla_primigenia <- 256931
PARAM$FT$semillerio <- 15  # cantidad de semillas

PARAM$experimento <- 9509
PARAM$dataset <- "analistasr_competencia_2026.csv.gz"

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp/WF9509_PREDICCIONES")
#experimento_folder <- paste0("WF_PREDICCIONES", PARAM$experimento) #experimento_folder <- paste0("WF", PARAM$experimento)
#dir.create(experimento_folder, showWarnings=FALSE)
#setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))
getwd()   # verifico que quedó bien

In [ ]:
# leo los 5 archivos de predicciones generados previamente con distintas semillas

#archivos <- c("prediccion1.txt", "prediccion2.txt", "prediccion3.txt",
#              "prediccion4.txt", "prediccion5.txt")
archivos <- c("prediccion1.txt", "prediccion2.txt") # quedan solo 2 porque los demás no terminaron de ejecutarse antes de medianoche

# leo solo cliente y prob de cada archivo, y renombro prob según el archivo
lista_tb <- lapply(seq_along(archivos), function(i) {
  tb <- fread(archivos[i], select = c("numero_de_cliente", "prob"))
  setnames(tb, "prob", paste0("prob_archivo_", i))
  tb
})

# uno todo por cliente (all = TRUE conserva clientes que falten en algún archivo)
tb_prediccion <- Reduce(
  function(a, b) merge(a, b, by = "numero_de_cliente", all = TRUE),
  lista_tb
)

# probabilidad promedio por cliente (fila)
cols_prob <- grep("^prob_archivo_", names(tb_prediccion), value = TRUE)
tb_prediccion[, prob := rowMeans(.SD, na.rm = TRUE), .SDcols = cols_prob]
setcolorder(tb_prediccion, c("numero_de_cliente", "prob"))

# ordeno por probabilidad decreciente
setorder(tb_prediccion, -prob)

# guardo
fwrite(tb_prediccion, "predicciones_combinadas.txt", sep = "\t")

head(tb_prediccion)


#### Kaggle Competition Submit

In [ ]:
Genero las salidas y hago los submits a Kaggles

In [ ]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

PARAM$kaggle$competencia <- "labo-1-virtual-2026-analista-sr"
PARAM$kaggle$cortes <- seq(9000, 13500, by = 500)

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle")

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0(
    "./kaggle/KA",
    PARAM$experimento, "_",
    "s", PARAM$FT$semillerio, "_",
    envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=", PARAM$semilla_primigenia,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  Sys.sleep(30)
  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  cat(salida, "\n")
}

In [ ]:
PARAM$tiempos$fin_notebook <- format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file="PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")